# Kredit tasdiqlash modeli — o'qitish noutbooki

**Maqsad:** ariza beruvchi ma'lumotlariga qarab kredit tasdiqlanishini (`Y` / `N`) bashorat qilish.

**Tartib:**
1. Kutubxonalar va sozlamalar
2. Ma'lumotni yuklash
3. Ma'lumotni o'rganish
4. Train / test bo'lish
5. Preprocessing (pipeline ichida)
6. Modellar va giperparametr qidiruvi
7. Modellarni solishtirish va tanlash
8. Test to'plamida yakuniy baholash
9. Saqlash
10. Saqlangan modelni tekshirish

**Kiruvchi fayl:** `loan_status.csv` &nbsp;|&nbsp; **Chiquvchi fayllar:** `final_model.pkl`, `LabelEncoder.pkl` (`predict.ipynb` bilan mos).

## 1. Kutubxonalar va sozlamalar

In [1]:
import pickle
import time
import warnings

import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV, RepeatedStratifiedKFold,
                                     StratifiedKFold, cross_validate, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

# ---------- Sozlamalar: o'zgartirish kerak bo'lsa faqat shu yerni tahrirlang ----------
DATA_PATH = "loan_status.csv"
SEED = 42
TEST_SIZE = 0.2
SCORING = "roc_auc"      # giperparametr qidiruvi va model tanlash uchun yagona metrika
CV_FOLDS = 5
REFIT_ON_ALL = False     # True: saqlashdan oldin modelni train+test ning hammasida qayta o'qitish
MANUAL_CHOICE = None     # masalan "SVC"; None bo'lsa model avtomatik tanlanadi

# scikit-learn 1.9+ da `SVC(probability=True)` eskirgan deb ogohlantiradi (hozircha ishlaydi). Faqat shu xabar yashiriladi.
warnings.filterwarnings("ignore", message="The `probability` parameter was deprecated")

print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")

pandas 3.0.5 | numpy 2.3.5 | scikit-learn 1.8.0


## 2. Ma'lumotni yuklash

Ustun nomlari o'zbekchaga **aniq xarita (dict)** orqali o'zgartiriladi — ro'yxat tartibiga tayanilmaydi,
shuning uchun CSV ustunlari tartibi o'zgarsa ham xato bo'lmaydi.
Nomlar `predict.ipynb` va saqlangan model bilan bir xil qoldirilgan.

In [2]:
USTUN_NOMLARI = {
    "Gender": "Jinsi",
    "Married": "Uylangan",
    "Dependents": "Qarindoshlari soni",          # aslida: qaramog'idagilar soni (Dependents)
    "Education": "Ma'lumoti",
    "Self_Employed": "O'zini o'zi ish bilan ta'minlagan",
    "ApplicantIncome": "Ariza beruvchining daromadi",
    "CoapplicantIncome": "Hamkorlikdagi ariza beruvchining daromadi",
    "LoanAmount": "Kredit miqdori",               # ming birlikda (trainda 9 ... 700)
    "Loan_Amount_Term": "Kredit miqdorining muddati",  # oy hisobida
    "Credit_History": "Kredit Tarixi",
    "Property_Area": "Mulk Maydoni",
    "Loan_Status": "Loan_Holati",
}
TARGET = "Loan_Holati"

raw = pd.read_csv(DATA_PATH)
yetmayotgan = set(USTUN_NOMLARI) - set(raw.columns)
assert not yetmayotgan, f"CSV da bu ustunlar yo'q: {yetmayotgan}"

df = raw.drop(columns="Loan_ID").rename(columns=USTUN_NOMLARI)
df.head()

,Jinsi,Uylangan,Qarindoshlari soni,Ma'lumoti,O'zini o'zi ish bilan ta'minlagan,Ariza beruvchining daromadi,Hamkorlikdagi ariza beruvchining daromadi,Kredit miqdori,Kredit miqdorining muddati,Kredit Tarixi,Mulk Maydoni,Loan_Holati
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


## 3. Ma'lumotni o'rganish

Bu bosqichda hech narsa o'zgartirilmaydi — faqat ko'riladi.

In [3]:
print("Shakl:", df.shape, "| to'liq takrorlangan qatorlar:", df.duplicated().sum())

tahlil = pd.DataFrame({
    "tur": df.dtypes.astype(str),
    "bo'sh": df.isna().sum(),
    "bo'sh %": (df.isna().mean() * 100).round(1),
    "noyob": df.nunique(),
})
display(tahlil)

print("\nMaqsad o'zgaruvchisi (ulush):")
print(df[TARGET].value_counts(normalize=True).round(3).to_string())

print("\nKredit tarixi bo'yicha tasdiqlanish ulushi:")
display(pd.crosstab(df["Kredit Tarixi"], df[TARGET], normalize="index", dropna=False).round(3))

display(df.describe().T.round(1))

Shakl: (614, 12) | to'liq takrorlangan qatorlar: 0


,tur,bo'sh,bo'sh %,noyob
Jinsi,str,13,2.1,2
Uylangan,str,3,0.5,2
Qarindoshlari soni,str,15,2.4,4
Ma'lumoti,str,0,0.0,2
O'zini o'zi ish bilan ta'minlagan,str,32,5.2,2
Ariza beruvchining daromadi,int64,0,0.0,505
Hamkorlikdagi ariza beruvchining daromadi,float64,0,0.0,287
Kredit miqdori,float64,22,3.6,203
Kredit miqdorining muddati,float64,14,2.3,10
Kredit Tarixi,float64,50,8.1,2



Maqsad o'zgaruvchisi (ulush):
Loan_Holati
Y    0.687
N    0.313

Kredit tarixi bo'yicha tasdiqlanish ulushi:


Loan_Holati,N,Y
Kredit Tarixi,,
0.0,0.921,0.079
1.0,0.204,0.796
NaN,0.260,0.740


,count,mean,std,min,25%,50%,75%,max
Ariza beruvchining daromadi,614.0,5403.5,6109.0,150.0,2877.5,3812.5,5795.0,81000.0
Hamkorlikdagi ariza beruvchining daromadi,614.0,1621.2,2926.2,0.0,0.0,1188.5,2297.2,41667.0
Kredit miqdori,592.0,146.4,85.6,9.0,100.0,128.0,168.0,700.0
Kredit miqdorining muddati,600.0,342.0,65.1,12.0,360.0,360.0,360.0,480.0
Kredit Tarixi,564.0,0.8,0.4,0.0,1.0,1.0,1.0,1.0


**Kuzatuvlar**
- Sinflar nomutanosib: ~69% `Y`. Hamma ariza `Y` deb javob beradigan "ahmoq" model ham ~0.69 accuracy oladi — shuning uchun natijalar shu baseline bilan solishtiriladi va faqat accuracy ga tayanilmaydi.
- `Kredit Tarixi` eng kuchli belgi: tarixi 1 bo'lganlarning ~80% i, 0 bo'lganlarning atigi ~8% i tasdiqlangan.
- Daromadlar va kredit miqdorida kuchli chekka qiymatlar (outlier) bor.
- Bo'sh qiymatlar bor, lekin ular **bu yerda to'ldirilmaydi** — 5-bo'limda pipeline ichida to'ldiriladi.

## 4. Train / test bo'lish

Bo'lish **birinchi** bajariladi, to'ldirish/masshtablash/kodlash esa faqat train'dan o'rganiladi (data leakage yo'q).
`stratify` — ikkala qismda `Y/N` ulushi bir xil bo'lishi uchun.

In [4]:
X = df.drop(columns=TARGET)
y_raw = df[TARGET]

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y_raw, test_size=TEST_SIZE, random_state=SEED, stratify=y_raw
)

# LabelEncoder faqat BIR marta, asl "N"/"Y" belgilar ustida o'qitiladi.
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

print("Sinflar:", dict(zip(le.classes_, le.transform(le.classes_))))
print(f"train: {X_train.shape} | test: {X_test.shape}")
print("train 'Y' ulushi:", y_train.mean().round(3), "| test 'Y' ulushi:", y_test.mean().round(3))

Sinflar: {'N': np.int64(0), 'Y': np.int64(1)}
train: (491, 11) | test: (123, 11)
train 'Y' ulushi: 0.686 | test 'Y' ulushi: 0.691


## 5. Preprocessing

Hammasi **bitta `ColumnTransformer` ichida**, shuning uchun:
- bo'sh qiymatlar to'ldirilishi ham modelning bir qismi — yangi ma'lumotda `NaN` bo'lsa model yiqilmaydi;
- to'ldirish qiymatlari (median / eng ko'p uchraydigan) faqat train'dan hisoblanadi;
- `handle_unknown="ignore"` — trainda ko'rilmagan yangi toifa kelsa xato bermaydi.

`Kredit Tarixi` 0/1 son bo'lsa ham **kategoriya sifatida** kodlanadi: sinovda uni oddiy son (0/1) sifatida
berganda `gamma=0.01` li SVC deyarli o'z kuchini yo'qotdi (CV accuracy ≈ 0.69, ya'ni baseline darajasi).

In [5]:
NUM_COLS = [
    "Ariza beruvchining daromadi",
    "Hamkorlikdagi ariza beruvchining daromadi",
    "Kredit miqdori",
    "Kredit miqdorining muddati",
]
CREDIT_COL = ["Kredit Tarixi"]
CAT_COLS = [c for c in X.columns if c not in NUM_COLS + CREDIT_COL]


def kategoriya_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])


def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUM_COLS),
        ("cat", kategoriya_pipeline(), CAT_COLS),
        ("credit", kategoriya_pipeline(), CREDIT_COL),
    ])


def make_pipeline(classifier):
    return Pipeline([("preprocessor", make_preprocessor()), ("classifier", classifier)])


print("Sonli      :", NUM_COLS)
print("Kategoriya :", CAT_COLS)
print("Kredit tarixi:", CREDIT_COL)

Sonli      : ['Ariza beruvchining daromadi', 'Hamkorlikdagi ariza beruvchining daromadi', 'Kredit miqdori', 'Kredit miqdorining muddati']
Kategoriya : ['Jinsi', 'Uylangan', 'Qarindoshlari soni', "Ma'lumoti", "O'zini o'zi ish bilan ta'minlagan", 'Mulk Maydoni']
Kredit tarixi: ['Kredit Tarixi']


## 6. Modellar va giperparametr qidiruvi

Hamma modellar **bir xil CV** (`StratifiedKFold`) va **bir xil metrika** (`SCORING`) bilan sozlanadi.
Qidiruv oralig'i eski noutbookdan kengaytirilgan: avval SVC ning eng yaxshi `C=1` va `gamma=0.01` qiymatlari
to'r (grid) ning **eng chetida** edi — bu yaxshisi shundan ham narida bo'lishi mumkinligini bildiradi.

In [6]:
cv_tuning = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

SEARCHES = {
    "SVC": GridSearchCV(
        make_pipeline(SVC(probability=True, random_state=SEED)),
        param_grid=[
            {"classifier__kernel": ["rbf"],
             "classifier__C": [0.03, 0.1, 0.3, 1, 3, 10, 30],
             "classifier__gamma": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]},
            {"classifier__kernel": ["linear"],
             "classifier__C": [0.03, 0.1, 0.3, 1, 3, 10, 30]},
        ],
        scoring=SCORING, cv=cv_tuning, n_jobs=-1,
    ),
    "KNN": GridSearchCV(
        make_pipeline(KNeighborsClassifier()),
        param_grid={
            "classifier__n_neighbors": [5, 7, 9, 11, 15, 21, 31],
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
        },
        scoring=SCORING, cv=cv_tuning, n_jobs=-1,
    ),
    "XGBoost": RandomizedSearchCV(
        make_pipeline(XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=1)),
        param_distributions={
            "classifier__n_estimators": randint(100, 600),
            "classifier__learning_rate": loguniform(0.01, 0.3),
            "classifier__max_depth": randint(2, 7),
            "classifier__min_child_weight": randint(1, 10),
            "classifier__subsample": uniform(0.6, 0.4),          # 0.6 – 1.0
            "classifier__colsample_bytree": uniform(0.6, 0.4),   # 0.6 – 1.0
            "classifier__gamma": uniform(0, 5),
            "classifier__reg_alpha": loguniform(1e-3, 10),
            "classifier__reg_lambda": loguniform(1e-2, 10),
        },
        n_iter=40, scoring=SCORING, cv=cv_tuning, n_jobs=-1, random_state=SEED,
    ),
    "CatBoost": RandomizedSearchCV(
        make_pipeline(CatBoostClassifier(verbose=0, random_state=SEED, thread_count=1,
                                         allow_writing_files=False)),
        param_distributions={
            "classifier__iterations": randint(100, 600),
            "classifier__learning_rate": loguniform(0.01, 0.2),
            "classifier__depth": randint(3, 7),
            "classifier__l2_leaf_reg": loguniform(1, 20),
            "classifier__random_strength": loguniform(0.1, 10),
        },
        n_iter=20, scoring=SCORING, cv=cv_tuning, n_jobs=-1, random_state=SEED,
    ),
}


def toza_parametrlar(params):
    """'classifier__' prefiksini olib tashlaydi va numpy sonlarini oddiy Python soniga o'giradi."""
    out = {}
    for k, v in params.items():
        v = v.item() if isinstance(v, np.generic) else v
        out[k.replace("classifier__", "")] = round(v, 4) if isinstance(v, float) else v
    return out


for nom, qidiruv in SEARCHES.items():
    t0 = time.time()
    qidiruv.fit(X_train, y_train)
    print(f"{nom:9s} CV {SCORING} = {qidiruv.best_score_:.4f}  ({time.time() - t0:.0f} s)")
    print(f"          {toza_parametrlar(qidiruv.best_params_)}")

SVC       CV roc_auc = 0.7489  (18 s)
          {'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}
KNN       CV roc_auc = 0.7380  (2 s)
          {'metric': 'manhattan', 'n_neighbors': 31, 'weights': 'distance'}
XGBoost   CV roc_auc = 0.7716  (5 s)
          {'colsample_bytree': 0.9633, 'gamma': 1.1978, 'learning_rate': 0.0164, 'max_depth': 5, 'min_child_weight': 6, 'n_estimators': 515, 'reg_alpha': 0.2348, 'reg_lambda': 0.0175, 'subsample': 0.7479}
CatBoost  CV roc_auc = 0.7610  (21 s)
          {'depth': 5, 'iterations': 535, 'l2_leaf_reg': 17.2547, 'learning_rate': 0.0896, 'random_strength': 1.5751}


**Eng yaxshi modellarni tayyorlash.** `RandomizedSearchCV` parametrlarni `np.float64` / `np.int64` ko'rinishida qaytaradi;
ba'zi kutubxonalar (masalan CatBoost) bunday qiymat bilan `clone()` da xato beradi.
Shuning uchun parametrlar oddiy Python soniga o'girilib, model train'da qayta o'qitiladi.

In [7]:
def numpy_tozala(params):
    return {k: (v.item() if isinstance(v, np.generic) else v) for k, v in params.items()}


def eng_yaxshi_model(qidiruv):
    model = clone(qidiruv.estimator).set_params(**numpy_tozala(qidiruv.best_params_))
    return model.fit(X_train, y_train)


TAYYOR = {nom: eng_yaxshi_model(q) for nom, q in SEARCHES.items()}

for nom, model in TAYYOR.items():
    clone(model)   # clone() xatosiz ishlashi shart (cross_validate va REFIT_ON_ALL ishlatadi)
print("Tayyor modellar:", list(TAYYOR))

Tayyor modellar: ['SVC', 'KNN', 'XGBoost', 'CatBoost']


## 7. Modellarni solishtirish va tanlash

Sozlangan modellar **takroriy stratified CV** (5 fold × 3 marta, boshqa `random_state`) bilan bir necha metrikada baholanadi.
`Dummy` — doim eng ko'p uchraydigan sinfni aytadigan baseline.

> Eslatma: bu baholash ham train ma'lumotida bajarilgan va giperparametrlar shu ma'lumotda tanlangan, shuning uchun
> ko'rsatkichlar biroz optimistik. Haqiqiy ko'rsatkich — 8-bo'limdagi test natijasi.

In [8]:
cv_eval = RepeatedStratifiedKFold(n_splits=CV_FOLDS, n_repeats=3, random_state=SEED + 1)
METRIKALAR = ["roc_auc", "accuracy", "f1", "f1_macro"]

nomzodlar = dict(TAYYOR)
nomzodlar["Dummy (baseline)"] = make_pipeline(DummyClassifier(strategy="most_frequent"))

qatorlar = []
for nom, model in nomzodlar.items():
    r = cross_validate(model, X_train, y_train, cv=cv_eval, scoring=METRIKALAR, n_jobs=-1)
    qatorlar.append({
        "Model": nom,
        "ROC AUC": r["test_roc_auc"].mean(),
        "ROC AUC std": r["test_roc_auc"].std(),
        "Accuracy": r["test_accuracy"].mean(),
        "F1 (Y)": r["test_f1"].mean(),
        "F1 macro": r["test_f1_macro"].mean(),
        "Fit vaqti (s)": r["fit_time"].mean(),
    })

taqqos = pd.DataFrame(qatorlar).set_index("Model").sort_values("ROC AUC", ascending=False)
display(taqqos.round(4))

,ROC AUC,ROC AUC std,Accuracy,F1 (Y),F1 macro,Fit vaqti (s)
Model,,,,,,
XGBoost,0.7652,0.0477,0.7767,0.8528,0.6924,0.2839
SVC,0.7484,0.0406,0.6864,0.8140,0.4070,0.1035
CatBoost,0.7351,0.0610,0.7564,0.8343,0.6839,2.7061
KNN,0.7331,0.0529,0.7502,0.8439,0.6084,0.0375
Dummy (baseline),0.5000,0.0000,0.6864,0.8140,0.4070,0.0291


In [9]:
# Tanlash qoidasi: Dummy dan tashqari modellar ichida eng yuqori CV ROC AUC.
raqobatchilar = taqqos.drop(index="Dummy (baseline)")
best_name = MANUAL_CHOICE or raqobatchilar.index[0]
print("Tanlangan model:", best_name, "(qo'lda)" if MANUAL_CHOICE else "(avtomatik)")

if not MANUAL_CHOICE and len(raqobatchilar) > 1:
    farq = raqobatchilar["ROC AUC"].iloc[0] - raqobatchilar["ROC AUC"].iloc[1]
    std = raqobatchilar["ROC AUC std"].iloc[0]
    print(f"1- va 2-o'rin ROC AUC farqi: {farq:.4f} | fold'lar bo'yicha std: {std:.4f}")
    if farq < std:
        print("Farq fold'lar orasidagi tebranishdan kichik: yetakchi modellar amalda teng kuchli.\n"
              "Bunda soddaroq/tezroq modelni MANUAL_CHOICE orqali tanlash ham oqilona.")

Tanlangan model: XGBoost (avtomatik)
1- va 2-o'rin ROC AUC farqi: 0.0168 | fold'lar bo'yicha std: 0.0477
Farq fold'lar orasidagi tebranishdan kichik: yetakchi modellar amalda teng kuchli.
Bunda soddaroq/tezroq modelni MANUAL_CHOICE orqali tanlash ham oqilona.


**Jadvalni o'qishda ehtiyot bo'ling:** SVC ning Accuracy va F1 macro ko'rsatkichi `Dummy` bilan deyarli bir xil bo'lsa,
bu model 0.5 chegarada hamma arizani `Y` deb belgilayapti degani. ROC AUC bo'yicha sozlash `gamma` ni grid'ning eng chetiga
surishi mumkin: ROC AUC qaror chegarasiga bog'liq emas, shuning uchun bunday model "yaxshi" ko'rinadi, lekin `predict()` da foydasiz.
Shu sababli faqat bitta ustunga emas, Accuracy / F1 macro ga ham qarang.

## 8. Test to'plamida yakuniy baholash

Test to'plami **faqat shu yerda, bir marta** ishlatiladi.

In [10]:
final_model = TAYYOR[best_name]      # train'da o'qitilgan
dummy = nomzodlar["Dummy (baseline)"].fit(X_train, y_train)


def baholash(model, X_, y_):
    pred = model.predict(X_)
    proba = model.predict_proba(X_)[:, 1]
    return {
        "Accuracy": accuracy_score(y_, pred),
        "F1 (Y)": f1_score(y_, pred),
        "F1 macro": f1_score(y_, pred, average="macro"),
        "ROC AUC": roc_auc_score(y_, proba),
    }


natija = pd.DataFrame({
    f"{best_name} — train": baholash(final_model, X_train, y_train),
    f"{best_name} — test": baholash(final_model, X_test, y_test),
    "Dummy — test": baholash(dummy, X_test, y_test),
}).T
display(natija.round(4))

pred_test = final_model.predict(X_test)
print(classification_report(y_test, pred_test, target_names=[str(c) for c in le.classes_]))

cm = pd.DataFrame(
    confusion_matrix(y_test, pred_test),
    index=[f"haqiqiy {c}" for c in le.classes_],
    columns=[f"bashorat {c}" for c in le.classes_],
)
display(cm)

,Accuracy,F1 (Y),F1 macro,ROC AUC
XGBoost — train,0.8330,0.8886,0.7776,0.9168
XGBoost — test,0.8374,0.8889,0.7929,0.7830
Dummy — test,0.6911,0.8173,0.4087,0.5000


              precision    recall  f1-score   support

           N       0.82      0.61      0.70        38
           Y       0.84      0.94      0.89        85

    accuracy                           0.84       123
   macro avg       0.83      0.77      0.79       123
weighted avg       0.84      0.84      0.83       123



,bashorat N,bashorat Y
haqiqiy N,23,15
haqiqiy Y,5,80


**Natijani o'qish:**
- Test to'plami kichik (123 qator): bitta xato ≈ 0.8% accuracy. Test accuracy ni CV natijasi bilan birga baholang — ikkalasi orasidagi farq qisman tasodif bo'lishi mumkin.
- Train va test ROC AUC orasidagi katta farq ortiqcha moslashish (overfitting) belgisi; daraxtli modellarda `max_depth`, `n_estimators` ni kamaytirish yoki regulyarizatsiyani oshirish mumkin.
- `N` sinfi recall'i `Y` nikidan past: model rad etilishi kerak arizalarning bir qismini tasdiqlayapti. Bu xato narxi muhim bo'lsa, `class_weight` yoki chegarani sozlash kerak.

**Qaysi belgilar muhim?** Permutation importance: test to'plamida belgi ustuni aralashtirilganda ROC AUC qancha tushadi
(ya'ni model shu belgiga qanchalik tayanadi). Pipeline ichida bo'lgani uchun asl ustunlar nomi bilan chiqadi.

In [11]:
pi = permutation_importance(final_model, X_test, y_test, scoring="roc_auc",
                            n_repeats=30, random_state=SEED, n_jobs=-1)
muhimlik = (pd.DataFrame({"AUC pasayishi": pi.importances_mean, "std": pi.importances_std}, index=X_test.columns)
            .sort_values("AUC pasayishi", ascending=False))
display(muhimlik.round(4))

,AUC pasayishi,std
Kredit Tarixi,0.2804,0.0423
Mulk Maydoni,0.0144,0.0132
Uylangan,0.0087,0.0074
Hamkorlikdagi ariza beruvchining daromadi,0.0002,0.0136
O'zini o'zi ish bilan ta'minlagan,-0.0007,0.0007
Kredit miqdorining muddati,-0.0010,0.0006
Ma'lumoti,-0.0016,0.0083
Jinsi,-0.0021,0.0022
Qarindoshlari soni,-0.0025,0.0116
Ariza beruvchining daromadi,-0.0058,0.0215


## 9. Saqlash

Fayl nomlari avvalgidek — `predict.ipynb` o'zgarishsiz ishlaydi. `LabelEncoder.pkl` endi haqiqiy `N`/`Y` belgilarini saqlaydi.

> Pickle fayl scikit-learn versiyasiga bog'liq: modelni o'qitgan va yuklagan muhitda versiya bir xil bo'lishi kerak.

In [12]:
if REFIT_ON_ALL:
    saqlanadigan_model = clone(final_model).fit(X, le.transform(y_raw))
    print("Model train+test ning hammasida qayta o'qitildi (test ko'rsatkichlari bu modelga endi to'g'ri kelmaydi).")
else:
    saqlanadigan_model = final_model
    print("Model faqat train qismida o'qitilgan holda saqlanadi (8-bo'lim ko'rsatkichlari shu modelniki).")

with open("final_model.pkl", "wb") as f:
    pickle.dump(saqlanadigan_model, f)
with open("LabelEncoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Saqlandi: final_model.pkl, LabelEncoder.pkl | scikit-learn", sklearn.__version__)

Model faqat train qismida o'qitilgan holda saqlanadi (8-bo'lim ko'rsatkichlari shu modelniki).
Saqlandi: final_model.pkl, LabelEncoder.pkl | scikit-learn 1.8.0


## 10. Saqlangan modelni tekshirish

Fayllar diskdan qayta o'qiladi va `predict.ipynb` dagi **aynan o'sha** `predict` funksiyasi bilan sinab ko'riladi.

In [13]:
with open("final_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("LabelEncoder.pkl", "rb") as f:
    le_loaded = pickle.load(f)


def predict(data):          # predict.ipynb dagi funksiya bilan bir xil
    if isinstance(data, dict):
        df_ = pd.DataFrame([data])
    elif isinstance(data, list):
        df_ = pd.DataFrame(data)
    else:
        df_ = data.copy()

    pred = np.ravel(model.predict(df_)).astype(int)
    labels = le_loaded.inverse_transform(pred)
    result = pd.DataFrame({"prediction": labels})
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(df_)
        for i, cls in enumerate(le_loaded.classes_):
            result[f"proba_{cls}"] = proba[:, i].round(4)
    return result


# 1) Diskdan o'qilgan model xotiradagi bilan bir xil natija berishi shart
assert (model.predict(X_test) == saqlanadigan_model.predict(X_test)).all()
assert list(le_loaded.classes_) == ["N", "Y"]
print("Tekshiruv o'tdi: natijalar bir xil, belgilar:", list(le_loaded.classes_))

# 2) Test to'plamidan bir nechta qator
display(pd.concat([X_test.head(3).reset_index(drop=True),
                   predict(X_test.head(3)),
                   y_test_raw.head(3).reset_index(drop=True).rename("haqiqiy")], axis=1)
        .iloc[:, -4:])

Tekshiruv o'tdi: natijalar bir xil, belgilar: ['N', 'Y']


,prediction,proba_N,proba_Y,haqiqiy
0,N,0.9132,0.0868,N
1,Y,0.2919,0.7081,Y
2,Y,0.3368,0.6632,Y


In [14]:
# 3) Real oraliqdagi yangi ariza (Kredit miqdori — MING birlikda; muddat — oy hisobida)
yangi = {
    "Jinsi": "Male", "Uylangan": "No", "Qarindoshlari soni": "0", "Ma'lumoti": "Graduate",
    "O'zini o'zi ish bilan ta'minlagan": "Yes",
    "Ariza beruvchining daromadi": 5000, "Hamkorlikdagi ariza beruvchining daromadi": 1500,
    "Kredit miqdori": 128, "Kredit miqdorining muddati": 360,
    "Kredit Tarixi": 1, "Mulk Maydoni": "Urban",
}
display(predict(yangi))

# 4) Bo'sh qiymat va yangi toifa — model endi yiqilmaydi
chetki = {**yangi, "Kredit Tarixi": np.nan, "Qarindoshlari soni": "4", "Jinsi": np.nan}
display(predict(chetki))


# 5) Trainda ko'rilgan oraliqdan tashqaridagi sonli qiymatlarni ogohlantirish
def oraliqni_tekshir(data):
    d = pd.DataFrame([data]) if isinstance(data, dict) else pd.DataFrame(data)
    for c in NUM_COLS:
        lo, hi = X_train[c].min(), X_train[c].max()
        for v in d[c].dropna():
            if not lo <= v <= hi:
                print(f"OGOHLANTIRISH: '{c}' = {v} trainda ko'rilgan oraliqdan [{lo:g}; {hi:g}] tashqarida — natija ishonchsiz.")


# predict.ipynb dagi eski misol: Kredit miqdori=5000, muddat=40
eski_misol = {**yangi, "Kredit miqdori": 5000, "Kredit miqdorining muddati": 40,
              "Ariza beruvchining daromadi": 10000, "Hamkorlikdagi ariza beruvchining daromadi": 10000}
oraliqni_tekshir(eski_misol)
display(predict(eski_misol))

,prediction,proba_N,proba_Y
0,Y,0.2485,0.7515


,prediction,proba_N,proba_Y
0,Y,0.2291,0.7709


OGOHLANTIRISH: 'Kredit miqdori' = 5000 trainda ko'rilgan oraliqdan [9; 700] tashqarida — natija ishonchsiz.


,prediction,proba_N,proba_Y
0,Y,0.4834,0.5166


## Eski noutbookka nisbatan nimalar tuzatildi

| # | Eski holat | Muammo | Hozir |
|---|---|---|---|
| 1 | `df[c] = df[c].fillna(..., inplace=True)` | Zanjirli inplace: pandas `ChainedAssignmentError` ogohlantirishini beradi, hujjatga ko'ra `inplace=True` `None` qaytaradi — ustun natijasi ishonchsiz | `SimpleImputer` pipeline ichida |
| 2 | To'ldirish train/test bo'lishdan **oldin** | Data leakage (median/mode butun ma'lumotdan) | Faqat train'dan o'rganiladi |
| 3 | `df['Qarindoshlari soni'] == 0` | Ustun satr (`'0'`), solishtirish songa — hech qachon `True` emas, shuning uchun `Uylangan` bo'shlari doim `'Yes'` bo'lgan | Alohida heuristika olib tashlandi, eng ko'p uchraydigan qiymat |
| 4 | `LabelEncoder` ikki marta o'qitilgan | Saqlangan `classes_ = [0, 1]` → `predict.ipynb` `N/Y` o'rniga `0/1` qaytargan | `LabelEncoder` bir marta, asl `N/Y` da |
| 5 | Pipeline'da imputer yo'q, `handle_unknown="error"` | Bo'sh qiymat yoki yangi toifada saqlangan model `ValueError` bilan yiqiladi | Imputer + `handle_unknown="ignore"` |
| 6 | Tanlash kodi `KNN` deb chiqargan, saqlangan esa qo'lda yozilgan `SVC` | Natija va saqlangan model bir-biriga zid | Tanlash qoidasi kodda, bitta metrika |
| 7 | Qidiruv `accuracy` da, tanlash `ROC AUC` da | Ikki xil mezon | Hammasi `SCORING` bilan |
| 8 | SVC grid: eng yaxshi `C=1`, `gamma=0.01` — grid chetida | Yaxshiroq qiymat grid'dan tashqarida qolishi mumkin | Grid ikki tomonga kengaytirildi |
| 9 | `train_test_split` `stratify`siz | Kichik test (123 qator)da sinf ulushi og'ishi mumkin | `stratify=y` |
| 10 | Baseline yo'q; `Transformer` global o'zgartirilgan; takroriy va bo'sh kataklar | Natijani baholash qiyin, qayta ishlatish xavfli | `Dummy` baseline, `make_preprocessor()` har safar yangi, ortiqcha kataklar yo'q |

**Keyingi qadamlar (ixtiyoriy):** `N` sinfi recall'i past bo'lsa (bank uchun noto'g'ri tasdiqlash qimmatroq bo'lishi mumkin) —
`class_weight="balanced"` yoki qaror chegarasini (threshold) sozlash; `daromad` ustunlariga `log1p`; `jami daromad` va oylik to'lov kabi yangi belgilar.

In [30]:
data = {
    'Jinsi': 'Female',
    "Uylangan": 'No',
    "Qarindoshlari soni": '1',
    "Ma'lumoti": 'Not Graduate',
    "O'zini o'zi ish bilan ta'minlagan": "Yes",
    "Ariza beruvchining daromadi": 1000,
    "Hamkorlikdagi ariza beruvchining daromadi": 1000,
    "Kredit miqdori": 50000,
    "Kredit miqdorining muddati": 240,
    "Kredit Tarixi": 1,
    "Mulk Maydoni": "Urban"
}

print(predict(data))

  prediction  proba_N  proba_Y
0          N   0.6571   0.3429
